In [1]:
import sys
sys.path.insert(0, '/home/alexstan/Documents/python-venv3.8/lib/python3.8/site-packages')
sys.path.insert(0, "/home/alexstan/Documents/PhD/GitHub/adaptive_sampling")
print(sys.path)

['/home/alexstan/Documents/PhD/GitHub/adaptive_sampling', '/home/alexstan/Documents/python-venv3.8/lib/python3.8/site-packages', '/home/alexstan/Documents/PhD/GitHub/adaptive_sampling/tutorials/HRD_TBLite', '/usr/lib/python38.zip', '/usr/lib/python3.8', '/usr/lib/python3.8/lib-dynload', '', '/home/alexstan/Documents/python-venv3.8/lib/python3.8/site-packages']


In [2]:
import sys, os
import json 
import networkx as nx
from collections import defaultdict
from ast import literal_eval
import numpy as np
from adaptive_sampling.processing_tools.exploration import pattern_mining, extract_tools, refinement_tools

In [3]:
# first perform the pattern mining (here shown for a single simulation collection)

work_dir = "/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K"

os.chdir(work_dir)
pattern_mining.find_freq_patterns(f"{work_dir}/reactions_lists", "rct_39K", 2)

Number of simulations found: 24


In [4]:
# then extract the raw trajectories for refinement

# the complete reactions are all patterns, 
# while rct_indices include only the pattern indices which are over the threshold
reactions, reactions_dict, complete_reactions, complete_reactions_dict, rct_indices = extract_tools.get_reaction_patterns(f"{work_dir}/rct_39K.out.json", freq_threshold=10) # adjust freq threshold to a min freq of patterns appearing in simulations

extract_tools.extract_reactions(f"{work_dir}",
                 complete_reactions,
                 rct_indices) # we extract only those over the threshold

Frequency of pattern # 0: 23
Pattern: [[['O=C=O', '[H]N([H])[H]'], ['[H][N+]([H])([H])[C-](=O)=O'], [1, 1], [1]], [['[H][N+]([H])([H])[C-](=O)=O'], ['O=C=O', '[H]N([H])[H]'], [1], [1, 1]]]
[[['O=C=O', '[H]N([H])[H]'], ['[H][N+]([H])([H])[C-](=O)=O']], [['[H][N+]([H])([H])[C-](=O)=O'], ['O=C=O', '[H]N([H])[H]']]] 

Frequency of pattern # 1: 11
Pattern: [[['[H]N([H])C(=O)OC(=O)[O-]'], ['[H]N([H])C(=O)O[C-](=O)=O'], [1], [1]]]
[[['[H]N([H])C(=O)OC(=O)[O-]'], ['[H]N([H])C(=O)O[C-](=O)=O']]] 

Directory already exists: extracted
reactions_list_000.json
reactions_list_001.json
reactions_list_002.json
reactions_list_003.json
reactions_list_004.json
reactions_list_005.json
reactions_list_006.json
reactions_list_007.json


KeyboardInterrupt: 

In [4]:
os.chdir("extracted/")
print(os.getcwd())
#os.chdir("..")

/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted


In [ ]:
# opt and freq directly with orca

import shutil
from pathlib import Path
from ase.calculators.orca import OrcaProfile, ORCA
profile = OrcaProfile(command="/home/alexstan/orca_6_1_0/orca")
calc = ORCA(profile=profile, orcasimpleinput='wb97x-3c opt freq', orcablocks='%freq  Temp  39 end')

current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    Path("start_opt").mkdir(exist_ok=True)
    Path("end_opt").mkdir(exist_ok=True)

    for file in dir.glob("pattern*.xyz"):
        extract_tools.extract_reactant_product(file, "start.xyz", "end.xyz")
        shutil.move("start.xyz", "start_opt/start.xyz")
        shutil.move("end.xyz", "end_opt/end.xyz")

    for mulliken_file in dir.glob("charge_mult*"):
        name = mulliken_file.name
        shutil.copy(mulliken_file, f"start_opt/{name}")
        shutil.copy(mulliken_file, f"end_opt/{name}")

    os.chdir("start_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="start.xyz")

    os.chdir("../end_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="end.xyz")

os.chdir("../..")
refinement_tools.find_minima()

/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/pattern0_traj_sim000_1_1
/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/pattern0_traj_sim000_2_2


In [5]:
os.chdir('MIN_FOUND')
os.getcwd()

'/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/MIN_FOUND'

In [6]:
# copy gsm_input and ostart to MIN_FOUND!
import shutil
from pathlib import Path
current_path = Path(os.getcwd())
for i, dir in enumerate(current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    shutil.copy('../ostart', 'ostart')
    shutil.copy('../gsm_input', 'gsm_input')
    for file in dir.glob("pattern*.xyz"):
        refinement_tools.rearrange_top(file)

    print(i)
    refinement_tools.de_gsm(add_args_file='gsm_input', out_file='log', index=i)



/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/MIN_FOUND/pattern0_traj_sim000_1_1
#========================================================#
#|                    building bonds                    |#
#========================================================#
None
 In build bonds
#========================================================#
#|                    building bonds                    |#
#========================================================#
None
 In build bonds
0
Charge: 0
Multiplicity: 1
/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/MIN_FOUND/pattern0_traj_sim000_2_2
#========================================================#
#|                    building bonds                    |#
#========================================================#
None
 In build bonds
#========================================================#
#|                    building bonds           

In [8]:
os.chdir('../')
os.getcwd()

'/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/MIN_FOUND'

In [ ]:
refinement_tools.find_viable_rcts() # fix this first so that it copies evth into ts_opt correctly!!

pattern0_traj_sim000_1_1/log
<re.Match object; span=(1225627, 1225645), match=' TS energy: 6.1320'>
6.132
pattern0_traj_sim000_1_1
6.132
Moving pattern0_traj_sim000_1_1 to VIABLE_RCTS
pattern0_traj_sim000_2_2/log
Total matching directories: 1


In [ ]:
# opt and freq directly with orca

import shutil
from pathlib import Path
from ase.calculators.orca import OrcaProfile, ORCA
profile = OrcaProfile(command="/home/alexstan/orca_6_1_0/orca")
calc = ORCA(profile=profile, orcasimpleinput='wb97x-3c opt freq', orcablocks='%freq  Temp  39 end')

current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    Path("start_opt").mkdir(exist_ok=True)
    Path("end_opt").mkdir(exist_ok=True)

    for file in dir.glob("pattern*.xyz"):
        extract_tools.extract_reactant_product(file, "start.xyz", "end.xyz")
        shutil.move("start.xyz", "start_opt/start.xyz")
        shutil.move("end.xyz", "end_opt/end.xyz")

    for mulliken_file in dir.glob("charge_mult*"):
        name = mulliken_file.name
        shutil.copy(mulliken_file, f"start_opt/{name}")
        shutil.copy(mulliken_file, f"end_opt/{name}")

    os.chdir("start_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="start.xyz")

    os.chdir("../end_opt")
    refinement_tools.ase_calc(calculator=calc, mol_file="end.xyz")

os.chdir("../..")
refinement_tools.find_minima()

In [7]:
# start the refinement by optimizing all reactants and products
import shutil
from pathlib import Path
from ase.calculators.orca import OrcaProfile, ORCA
profile = OrcaProfile(command="/home/alexstan/orca_6_1_0/orca")
calc = ORCA(profile=profile, orcasimpleinput='wb97x-3c engrad')

current_path = Path(os.getcwd())
for dir in (current_path.glob("pattern*")):
    print(dir)
    os.chdir(dir)
    Path("start_opt").mkdir(exist_ok=True)
    Path("end_opt").mkdir(exist_ok=True)

    for file in dir.glob("pattern*.xyz"):
        extract_tools.extract_reactant_product(file, "start.xyz", "end.xyz")
        shutil.move("start.xyz", "start_opt/start.xyz")
        shutil.move("end.xyz", "end_opt/end.xyz")

    for mulliken_file in dir.glob("charge_mult*"):
        name = mulliken_file.name
        shutil.copy(mulliken_file, f"start_opt/{name}")
        shutil.copy(mulliken_file, f"end_opt/{name}")

    os.chdir("start_opt")
    refinement_tools.sella_opt(calculator=calc, mol_file="start.xyz", order=0)

    os.chdir("../end_opt")
    refinement_tools.sella_opt(calculator=calc, mol_file="end.xyz", order=0)

os.chdir("../..")

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/pattern0_traj_sim000_1_1
Bad internals found!
Bad internals found!
Opt done!
Bad internals found!
Bad internals found!
Opt done!
/home/alexstan/Documents/PhD/4_Nanoreactor_Graph_Analysis/test_code/test_sim_data/test_39K/extracted/pattern0_traj_sim000_2_2
Bad internals found!
Bad internals found!
Opt done!
Opt done!
